# Fase 1 — Dados

Baixa e prepara os dois datasets (Construction Site Safety para detecção, COCO person para segmentação), gera splits reprodutíveis e roda a análise exploratória (EDA).

Ver `docs/relatorio-tecnico.md` no repositório para os resultados já obtidos com estes mesmos passos.

## Setup

In [ ]:
!pip install -q ultralytics opencv-python pandas pyarrow matplotlib pillow kaggle

from pathlib import Path

# Armazenamento persistente compartilhado entre os notebooks: monta o Google
# Drive e usa uma pasta fixa. Troque o caminho se preferir outra estrutura.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/vc-seguranca-trabalho')
except ImportError:
    # Execucao fora do Colab (teste local) - usa uma pasta local.
    PROJECT_DIR = Path('./vc-seguranca-trabalho').resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
ROOT = PROJECT_DIR
print("Diretorio do projeto:", ROOT)


### Token do Kaggle

O dataset "Construction Site Safety" é baixado do Kaggle e exige autenticação.
Gere um token em [kaggle.com/settings/api](https://www.kaggle.com/settings/api)
("Create New Token") e cole abaixo (fica só nesta sessão do Colab — não é salvo no notebook).

In [ ]:
import os

KAGGLE_TOKEN = ""  # cole aqui o token gerado em kaggle.com/settings/api

if KAGGLE_TOKEN:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / "access_token").write_text(KAGGLE_TOKEN)
    print("Token do Kaggle configurado.")
else:
    print("AVISO: configure KAGGLE_TOKEN acima antes de rodar a proxima celula.")


## 1. Baixar e amostrar o dataset de detecção (Construction Site Safety)

Baixa o dataset completo do Kaggle e seleciona um subconjunto de ~350 imagens por amostragem estratificada (seed=42), garantindo presença mínima de todas as 10 classes de EPI.

In [ ]:
import random
import shutil
import subprocess
import sys
import tempfile
from collections import defaultdict
from pathlib import Path

SEED = 42
TARGET_TOTAL = 350
MIN_PER_CLASS = 15

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]

KAGGLE_DATASET = "snehilsanyal/construction-site-safety-image-dataset-roboflow"
OUTPUT_ROOT = ROOT / "data" / "raw" / "construction-site-safety"


def download_kaggle_dataset(tmp_dir: Path) -> Path:
    """Baixa (uma unica vez) o dataset do Kaggle via CLI. Requer um token
    configurado em ~/.kaggle/access_token (celula acima)."""
    css_data_dir = tmp_dir / "css-data"
    if css_data_dir.exists():
        return css_data_dir

    token_path = Path.home() / ".kaggle" / "access_token"
    if not token_path.exists():
        raise RuntimeError(
            f"Token do Kaggle nao encontrado em {token_path}. "
            "Rode a celula 'Token do Kaggle' acima antes desta."
        )

    print("Baixando dataset 'Construction Site Safety' do Kaggle (~206MB)...")
    result = subprocess.run(
        [sys.executable, "-m", "kaggle", "datasets", "download",
         "-d", KAGGLE_DATASET, "-p", str(tmp_dir), "--unzip"],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            f"Falha ao baixar dataset do Kaggle (codigo {result.returncode}). "
            "Veja a mensagem de erro acima - geralmente e token invalido/expirado."
        )
    return css_data_dir


def read_classes_in_label(label_path: Path) -> set:
    classes = set()
    if not label_path.exists():
        return classes
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cls_id = int(line.split()[0])
            classes.add(cls_id)
    return classes


def collect_pool(source_root: Path):
    pool = []
    for split in ["train", "valid", "test"]:
        images_dir = source_root / split / "images"
        labels_dir = source_root / split / "labels"
        if not images_dir.exists():
            continue
        for img_path in images_dir.iterdir():
            if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            label_path = labels_dir / (img_path.stem + ".txt")
            classes = read_classes_in_label(label_path)
            pool.append((img_path, label_path, classes))
    return pool


random.seed(SEED)

tmp_dir = Path(tempfile.gettempdir()) / "css_kaggle_cache"
tmp_dir.mkdir(parents=True, exist_ok=True)
source_root = download_kaggle_dataset(tmp_dir)

pool = collect_pool(source_root)
print(f"Total de imagens disponiveis na fonte: {len(pool)}")

random.shuffle(pool)

selected = []
selected_paths = set()
class_counts = defaultdict(int)

# Passo 1: garantir cobertura minima de cada classe
for cls_id in range(len(CLASS_NAMES)):
    for img_path, label_path, classes in pool:
        if img_path in selected_paths:
            continue
        if cls_id in classes and class_counts[cls_id] < MIN_PER_CLASS:
            selected.append((img_path, label_path, classes))
            selected_paths.add(img_path)
            for c in classes:
                class_counts[c] += 1
        if class_counts[cls_id] >= MIN_PER_CLASS:
            break

# Passo 2: completar ate o total alvo com amostragem aleatoria (seed fixa)
for img_path, label_path, classes in pool:
    if len(selected) >= TARGET_TOTAL:
        break
    if img_path in selected_paths:
        continue
    selected.append((img_path, label_path, classes))
    selected_paths.add(img_path)
    for c in classes:
        class_counts[c] += 1

print(f"Subconjunto selecionado: {len(selected)} imagens (seed={SEED})")
print("Contagem de instancias-imagem por classe no subconjunto:")
for cls_id, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {class_counts.get(cls_id, 0)} imagens")

images_out = OUTPUT_ROOT / "images"
labels_out = OUTPUT_ROOT / "labels"
images_out.mkdir(parents=True, exist_ok=True)
labels_out.mkdir(parents=True, exist_ok=True)

copied = 0
for img_path, label_path, _ in selected:
    shutil.copy2(img_path, images_out / img_path.name)
    if label_path.exists():
        shutil.copy2(label_path, labels_out / label_path.name)
    copied += 1

print(f"Imagens+labels copiados: {copied}")
print("data.yaml (formato Ultralytics, com train/val/test) e escrito na secao 5, nao aqui.")


## 2. Baixar e amostrar o dataset de segmentação (COCO person)

Baixa as anotações COCO 2017 (val2017), filtra a categoria `person` com máscara, e baixa 300 imagens (seed=42). Fonte pública, não exige credenciais.

In [ ]:
import json
import random
import tempfile
import urllib.request
import zipfile
from pathlib import Path

SEED = 42
TARGET_IMAGES = 300
ANNOTATIONS_ZIP_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ANNOTATIONS_MEMBER = "annotations/instances_val2017.json"
OUTPUT_DIR = ROOT / "data" / "raw" / "coco-person"
IMAGES_DIR = OUTPUT_DIR / "images"
COCO_IMAGE_BASE_URL = "http://images.cocodataset.org/val2017/"


def download_and_extract_annotations(tmp_dir: Path) -> Path:
    """Baixa apenas o instances_val2017.json de dentro do zip de anotacoes do
    COCO (241MB), sem precisar manter o zip inteiro em disco depois."""
    extracted_path = tmp_dir / "instances_val2017.json"
    if extracted_path.exists():
        return extracted_path

    zip_path = tmp_dir / "annotations_trainval2017.zip"
    print("Baixando anotacoes do COCO (241MB, so uma vez)...")
    urllib.request.urlretrieve(ANNOTATIONS_ZIP_URL, zip_path)

    with zipfile.ZipFile(zip_path) as zf:
        zf.extract(ANNOTATIONS_MEMBER, tmp_dir)
    (tmp_dir / ANNOTATIONS_MEMBER).rename(extracted_path)
    zip_path.unlink()
    return extracted_path


random.seed(SEED)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

tmp_dir = Path(tempfile.gettempdir()) / "coco_annotations_cache"
tmp_dir.mkdir(parents=True, exist_ok=True)
annotations_path = download_and_extract_annotations(tmp_dir)

with open(annotations_path, "r", encoding="utf-8") as f:
    coco = json.load(f)

person_cat_id = next(c["id"] for c in coco["categories"] if c["name"] == "person")
images_by_id = {img["id"]: img for img in coco["images"]}

person_anns_by_image = {}
for ann in coco["annotations"]:
    if ann["category_id"] == person_cat_id and ann.get("iscrowd", 0) == 0:
        person_anns_by_image.setdefault(ann["image_id"], []).append(ann)

eligible_image_ids = [img_id for img_id, anns in person_anns_by_image.items() if len(anns) >= 1]
eligible_image_ids.sort()
random.shuffle(eligible_image_ids)

selected_ids = sorted(eligible_image_ids[:TARGET_IMAGES])
print(f"Imagens elegiveis (com pessoa e mascara): {len(eligible_image_ids)}")
print(f"Selecionadas (seed={SEED}): {len(selected_ids)}")

selected_images = [images_by_id[i] for i in selected_ids]
selected_annotations = [ann for i in selected_ids for ann in person_anns_by_image[i]]

subset = {
    "info": coco.get("info", {}),
    "licenses": coco.get("licenses", []),
    "categories": [c for c in coco["categories"] if c["id"] == person_cat_id],
    "images": selected_images,
    "annotations": selected_annotations,
}

subset_path = OUTPUT_DIR / "instances_person_subset.json"
with open(subset_path, "w", encoding="utf-8") as f:
    json.dump(subset, f)
print(f"Anotacoes filtradas salvas em: {subset_path}")

total_instances = len(selected_annotations)
print(f"Total de instancias de pessoa no subconjunto: {total_instances}")

downloaded = 0
for img in selected_images:
    dest = IMAGES_DIR / img["file_name"]
    if dest.exists():
        downloaded += 1
        continue
    url = COCO_IMAGE_BASE_URL + img["file_name"]
    try:
        urllib.request.urlretrieve(url, dest)
        downloaded += 1
    except Exception as e:
        print(f"Falha ao baixar {img['file_name']}: {e}")

print(f"Imagens baixadas: {downloaded}/{len(selected_images)}")


## 3. Gerar splits reprodutíveis (70/20/10)

Gera as listas de treino/validação/teste para as duas fontes, com seed=42.

In [ ]:
import random
from pathlib import Path

SEED = 42
RATIOS = {"train": 0.7, "val": 0.2, "test": 0.1}

SOURCES = {
    "construction-site-safety": ROOT / "data" / "raw" / "construction-site-safety" / "images",
    "coco-person": ROOT / "data" / "raw" / "coco-person" / "images",
}
SPLITS_ROOT = ROOT / "data" / "splits"


def split_list(files, ratios, seed):
    files = sorted(files)
    rng = random.Random(seed)
    rng.shuffle(files)
    n = len(files)
    n_train = int(n * ratios["train"])
    n_val = int(n * ratios["val"])
    return {
        "train": files[:n_train],
        "val": files[n_train : n_train + n_val],
        "test": files[n_train + n_val :],
    }


for source_name, images_dir in SOURCES.items():
    if not images_dir.exists():
        print(f"[aviso] pasta nao encontrada, pulando: {images_dir}")
        continue
    files = [p.name for p in images_dir.iterdir() if p.is_file()]
    splits = split_list(files, RATIOS, SEED)

    out_dir = SPLITS_ROOT / source_name
    out_dir.mkdir(parents=True, exist_ok=True)
    for split_name, split_files in splits.items():
        out_path = out_dir / f"{split_name}.txt"
        with open(out_path, "w", encoding="utf-8") as f:
            f.write("\n".join(split_files) + "\n")

    print(f"{source_name}: total={len(files)} "
          f"train={len(splits['train'])} val={len(splits['val'])} test={len(splits['test'])} "
          f"(seed={SEED})")


## 4. Análise exploratória (EDA)

Contagem de instâncias por classe, resolução e brilho médio (proxy de iluminação) do dataset de detecção.

In [ ]:
from collections import Counter
from pathlib import Path

from PIL import Image
import matplotlib.pyplot as plt

DATA_DIR = ROOT / "data" / "raw" / "construction-site-safety"
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"
FIGURES_DIR = ROOT / "reports" / "figures"

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]


def count_instances_per_class():
    counts = Counter()
    for label_path in LABELS_DIR.glob("*.txt"):
        with open(label_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                cls_id = int(line.split()[0])
                counts[cls_id] += 1
    return counts


def sample_resolutions_and_brightness(sample_size=100):
    image_paths = sorted(IMAGES_DIR.iterdir())[:sample_size]
    resolutions = []
    brightness = []
    for p in image_paths:
        try:
            with Image.open(p) as img:
                resolutions.append(img.size)
                gray = img.convert("L")
                pixels = list(gray.getdata())
                brightness.append(sum(pixels) / len(pixels))
        except Exception as e:
            print(f"[aviso] falha ao ler {p.name}: {e}")
    return resolutions, brightness


FIGURES_DIR.mkdir(parents=True, exist_ok=True)

counts = count_instances_per_class()
total_instances = sum(counts.values())
print("Instancias por classe:")
for cls_id, name in enumerate(CLASS_NAMES):
    n = counts.get(cls_id, 0)
    pct = (n / total_instances * 100) if total_instances else 0
    print(f"  {name}: {n} ({pct:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 5))
values = [counts.get(i, 0) for i in range(len(CLASS_NAMES))]
ax.bar(CLASS_NAMES, values, color="#4C72B0")
ax.set_ylabel("Numero de instancias")
ax.set_title("Instancias por classe - Construction Site Safety (subconjunto)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "css_instances_per_class.png", dpi=150)
plt.show()

resolutions, brightness = sample_resolutions_and_brightness(sample_size=150)
widths = [w for w, h in resolutions]
heights = [h for w, h in resolutions]

print(f"\nResolucao (amostra de {len(resolutions)} imagens):")
if widths:
    print(f"  largura: min={min(widths)} max={max(widths)} media={sum(widths)/len(widths):.0f}")
    print(f"  altura:  min={min(heights)} max={max(heights)} media={sum(heights)/len(heights):.0f}")
if brightness:
    print(f"  brilho medio (0-255): min={min(brightness):.1f} max={max(brightness):.1f} "
          f"media={sum(brightness)/len(brightness):.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=20, color="#55A868")
axes[0].set_title("Distribuicao de largura (px)")
axes[1].hist(brightness, bins=20, color="#C44E52")
axes[1].set_title("Distribuicao de brilho medio (proxy de iluminacao)")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "css_resolution_brightness.png", dpi=150)
plt.show()

max_class = max(counts, key=counts.get)
min_class = min(range(len(CLASS_NAMES)), key=lambda i: counts.get(i, 0))
ratio = counts[max_class] / max(counts.get(min_class, 1), 1)
print(f"\nDesbalanceamento: classe mais frequente = {CLASS_NAMES[max_class]} "
      f"({counts[max_class]} instancias); classe menos frequente = "
      f"{CLASS_NAMES[min_class]} ({counts.get(min_class, 0)} instancias); "
      f"razao aproximada = {ratio:.1f}x")
